# Segmentasi Instance Indoor — Evaluasi CPMK 3

NRP **5054241018**. Notebook Kaggle ini mencakup persiapan data, YOLO11n-seg,
Mask R-CNN ResNet-50 FPN, RF-DETR Seg Nano, eksperimen tiga kelas tersulit,
dan evaluasi. **Fokus tahap sekarang adalah dataset Roboflow dan perbandingan
tiga model pada semua kelas. Foto HP belum diambil dan belum digunakan.**

## Prosedur eksekusi

1. Tambahkan dataset Roboflow sebagai Input. Aktifkan GPU dan Internet di Kaggle.
2. Konfigurasi default siap menjalankan training: `MODE = "train"`, `MODEL = "all"`,
   `SCENARIO = "all_classes"`, `EPOCHS = 30`, dan `SMOKE_TEST = False`.
3. Jalankan **Run All**. Dataset audit dilakukan sebelum training.
4. Runner mengeksekusi **YOLO11n-seg → Mask R-CNN → RF-DETR Seg Nano** secara
   berurutan. Setiap model menggunakan subprocess dan virtual environment
   tersendiri dengan akses PyTorch bawaan Kaggle.
5. Checkpoint dan log disimpan per model. Ordinary error dicatat dan model
   berikutnya tetap dijalankan. KeyboardInterrupt menghentikan seluruh sequence.
6. Best checkpoint dipilih menggunakan validation metrics; `FINAL_TEST = True`
   menjalankan Roboflow test evaluation setelah training masing-masing model.
7. Runner melewati run yang lengkap dengan configuration signature yang sama.
   Jika terdapat checkpoint yang kompatibel, runner mencoba auto-resume.
8. Tabel perbandingan dan recovery archive tersedia setelah sequence berakhir.

Set `MODEL` menjadi `yolo11`, `maskrcnn`, atau `rfdetr` untuk menjalankan satu model.
Gunakan `MODE = "prepare"` jika hanya memerlukan dataset audit. Set `SMOKE_TEST = True`
untuk menguji sequence pada subset kecil sebelum training utama.

Tahap berikutnya: `MODE = "select_classes"` menentukan tiga kelas tersulit
dari validation, lalu tiga model dilatih pada `hard3`. Foto HP, domain shift, dan
analisis kegagalan dijalankan kemudian dengan `ENABLE_PERSONAL = True` setelah
foto dan anotasinya tersedia. Biarkan `ENABLE_PERSONAL = False` untuk tahap sekarang.

Notebook menjalankan satu model pada satu waktu. Semua model memakai
pretrained weights, sedangkan head disesuaikan dengan kelas dataset. `SMOKE_TEST`
menguji alur dengan data kecil; hasilnya dipisahkan dari eksperimen utama.
Metrik di notebook belum merupakan hasil eksperimen sebelum kode dijalankan.


## 1. Konfigurasi eksperimen


### 1.1 Runtime cell registry

Registrasi source cell digunakan untuk membentuk worker script dari kode yang ditampilkan di notebook. Jalankan notebook sebagai cell terpisah melalui Run All.


In [ ]:
CELL_REGISTRY = {}

def register_cell(cell_id, tag):
    shell = get_ipython()
    source = ""
    if getattr(shell, "kernel", None) is not None:
        source = shell.kernel.get_parent().get("content", {}).get("code", "")
    if not source:
        source = shell.history_manager.input_hist_raw[-1]
    lines = source.splitlines(keepends=True)
    expected = f"register_cell({cell_id!r}, {tag!r})"
    if not lines or lines[0].strip() != expected:
        raise RuntimeError("Source cell tidak sesuai. Jalankan notebook dengan cell terpisah.")
    CELL_REGISTRY[cell_id] = {"tag": tag, "source": "".join(lines[1:])}


### Ekspor progres tanpa menunggu seluruh notebook

Checkpoint ditulis setiap akhir epoch: YOLO `last.pt`, Mask R-CNN `last.pth`,
dan RF-DETR `native/last.ckpt`. Cell ekspor berikut hanya memakai library standar
Python, sehingga bisa dijalankan tanpa setup dataset atau GPU. Untuk membuat ZIP
secara manual, interupsi cell training, bukan menghentikan sesi Kaggle, lalu
jalankan tiga cell pada bagian ini. Pada kernel baru, jalankan Runtime cell registry
terlebih dahulu. Jangan pilih Run All untuk melakukan manual export.

Setiap kali cell training selesai, gagal, atau menerima KeyboardInterrupt,
notebook juga mencoba mengekspor checkpoint yang sudah selesai ditulis.
Progres epoch yang belum sempat disimpan tidak dijamin pulih. ZIP tetap berada
di disk sesi Kaggle: unduh ke laptop atau simpan versi dengan output agar
tersedia di luar sesi. Jika VM/kernel dimatikan paksa, blok penyelamatan Python
tidak dapat berjalan; hanya salinan yang sudah diunduh/disimpan yang dapat diandalkan.


### 1.2 Daftar recovery files

Temukan file checkpoint dan log yang akan diselamatkan. File sementara tidak dimasukkan ke arsip.


In [ ]:
register_cell('cell-006', 'recovery_files')
from pathlib import Path
import os
import zipfile

def recovery_files(folder):
    checkpoint_names = {"best.pt", "last.pt", "best.pth", "last.pth", "last.ckpt",
                        "checkpoint_best_regular.pth", "checkpoint_best_ema.pth",
                        "checkpoint_best_total.pth", "last_ema.pth"}
    metadata_suffixes = {".json", ".csv", ".txt", ".yaml", ".yml"}
    files = []
    for path in sorted(Path(folder).rglob("*")):
        if not path.is_file() or path.suffix in {".tmp", ".part"}:
            continue
        if path.name in checkpoint_names or path.suffix in metadata_suffixes or "tfevents" in path.name:
            files.append(path)
    return files


### 1.3 Recovery archive

Buat recovery archive secara atomik agar arsip lama tetap ada jika ekspor baru gagal.


In [ ]:
register_cell('cell-008', 'recovery_export')
def export_recovery(folder):
    folder = Path(folder).resolve()
    files = recovery_files(folder)
    if not any(p.suffix in {".pt", ".pth", ".ckpt"} for p in files):
        print("Belum ada checkpoint lengkap untuk diekspor:", folder)
        return None
    kaggle = Path("/kaggle/working").is_dir()
    destination = Path("/kaggle/working") if kaggle else folder.parent
    name = "_".join(folder.parts[-3:])
    target = destination / f"recovery_{name}.zip"
    partial = target.with_suffix(".zip.part")
    try:
        with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for path in files:
                before = path.stat()
                archive.write(path, arcname=str(Path(folder.name) / path.relative_to(folder)))
                after = path.stat()
                if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
                    raise RuntimeError(f"File berubah saat diekspor; ulangi setelah training diinterupsi: {path}")
        os.replace(partial, target)
    finally:
        if partial.exists():
            partial.unlink()
    print("Arsip pemulihan siap diunduh:", target)
    return target


### 1.4 Manual checkpoint export

Untuk ekspor manual, ubah EXPORT_RECOVERY_NOW menjadi True dan isi RECOVERY_RUN atau gunakan run terbaru.


In [ ]:
register_cell('cell-010', 'recovery_now')
EXPORT_RECOVERY_NOW = False
RECOVERY_RUN = ""
if EXPORT_RECOVERY_NOW:
    base = Path("/kaggle/working/cpmk3") if Path("/kaggle/working").is_dir() else Path.cwd() / "outputs/cpmk3"
    candidates = list((base / "artifacts").glob("*/*/*/run_config.json"))
    folder = Path(RECOVERY_RUN) if RECOVERY_RUN else (
        max(candidates, key=lambda p: p.stat().st_mtime).parent if candidates else None)
    if folder is None:
        print("Isi RECOVERY_RUN dengan folder run yang checkpoint-nya ingin diselamatkan.")
    else:
        RECOVERY_ARCHIVE = export_recovery(folder)


### 1.5 Konfigurasi eksperimen

Pilih pekerjaan, model, dan skenario kelas untuk run ini.


In [ ]:
register_cell('cell-012', 'config')
MODE = "train"
MODEL = "all"
MODEL_ORDER = ["yolo11", "maskrcnn", "rfdetr"]
SKIP_COMPLETED = True
AUTO_RESUME = True
CONTINUE_ON_ERROR = True
SCENARIO = "all_classes"
SMOKE_TEST = False
FINAL_TEST = True
ENABLE_PERSONAL = False
SEED = 42
EPOCHS = 30
RUN_LABEL = "run01"
RESUME_CHECKPOINT = ""
INFERENCE_CHECKPOINT = ""
if SMOKE_TEST:
    FINAL_TEST = False


### 1.6 Konfigurasi eksperimen

Gunakan folder utama dataset Kaggle yang berisi train, valid, dan test sesuai Input yang sudah ditambahkan.


In [ ]:
register_cell('cell-014', 'config')
RAW_DATASET = "/kaggle/input/datasets/mrmfrhann234/dataset-panoptic"
PERSONAL_DATASET = ""
PERSONAL_METADATA = ""
HARD3_FILE = ""
PREVIOUS_RESULTS = []
YOLO_BATCH = 4
MASKRCNN_BATCH = 2
RFDETR_BATCH = 2
YOLO_SIZE = 640
MASKRCNN_SHORT = 480
MASKRCNN_LONG = 640
RFDETR_SIZE = 312
SCORE_FLOOR = 0.001
MAX_DETECTIONS = 100
MIN_VALID_IMAGES = 5
MIN_TRAIN_IMAGES = 5


Resolusi RF-DETR Nano mengikuti ukuran bawaan 312, sedangkan kedua CNN memakai
batas resolusi di atas. Perbedaan preprocessing dan kapasitas model harus
dicantumkan dalam analisis; ini bukan eksperimen dengan komputasi identik.
Epoch dan batch merupakan titik awal, lalu dikunci setelah uji singkat.
Catat total waktu GPU setiap model, termasuk kedua skenario. Catatan PDF
menyebut maksimum dua jam GPU Colab per model jika akses GPU sulit.


### 1.7 Instalasi dependencies

Pasang paket analisis dan hanya framework tambahan yang diperlukan model terpilih.


In [ ]:
register_cell('cell-017', 'install')
import sys
import subprocess

packages = ["numpy", "pandas", "matplotlib", "pillow", "pycocotools", "pyyaml"]
needs_model = MODE in {"train", "predict", "diagnose"} and MODEL != "all"
if needs_model and MODEL == "yolo11":
    packages.append("ultralytics==8.4.215")
if needs_model and MODEL == "rfdetr":
    packages.append("rfdetr[train]==1.12.1")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


### 1.8 Inisialisasi library

Muat library umum dan validation pilihan konfigurasi.


In [ ]:
register_cell('cell-019', 'imports')
import copy
import csv
import gc
import gzip
import hashlib
import importlib.metadata
import io
import json
import random
import shutil
import time
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageEnhance
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

assert MODE in {"prepare", "train", "predict", "select_classes", "report", "diagnose"}
assert MODEL in {"all", "yolo11", "maskrcnn", "rfdetr"}
assert MODEL != "all" or MODE in {"train", "prepare", "select_classes", "report"}
assert MODEL_ORDER == ["yolo11", "maskrcnn", "rfdetr"]
assert SCENARIO in {"all_classes", "hard3"}
assert RUN_LABEL and all(c.isalnum() or c in "-_" for c in RUN_LABEL)
assert EPOCHS >= 1 and 0 <= SCORE_FLOOR < 1 and MAX_DETECTIONS == 100
random.seed(SEED)
np.random.seed(SEED)


### 1.9 Direktori kerja dan utilitas JSON

Siapkan direktori kerja tanpa mengubah dataset sumber.


In [ ]:
register_cell('cell-021', 'paths')
IS_KAGGLE = Path("/kaggle/input").is_dir()
WORK = Path("/kaggle/working/cpmk3") if IS_KAGGLE else Path.cwd() / "outputs/cpmk3"
PREPARED = WORK / "prepared"
REPORTS = WORK / "reports"
for folder in [WORK, PREPARED, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + ".tmp")
    try:
        partial.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        os.replace(partial, path)
    finally:
        if partial.exists():
            partial.unlink()

def fingerprint(data):
    raw = json.dumps(data, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


## 2. Membaca dan memeriksa dataset


### 2.1 Lokasi dataset

Cari satu dataset v5 dengan pembagian train, valid, dan test. Jika ditemukan lebih dari satu, isi RAW_DATASET.


In [ ]:
register_cell('cell-024', 'discover')
def locate_dataset():
    if RAW_DATASET:
        root = Path(RAW_DATASET)
    elif IS_KAGGLE:
        candidates = []
        for annotation in Path("/kaggle/input").rglob("_annotations.coco.json"):
            root = annotation.parent.parent
            if annotation.parent.name != "train":
                continue
            if all((root / s / "_annotations.coco.json").exists() for s in ["train", "valid", "test"]):
                candidates.append(root)
        candidates = list(dict.fromkeys(candidates))
        exports = [p for p in candidates if (p / "README.roboflow.txt").exists()]
        if exports:
            candidates = exports
        if len(candidates) != 1:
            raise ValueError(f"Isi RAW_DATASET. Kandidat: {candidates}")
        root = candidates[0]
    else:
        candidates = [Path("dataset"), Path("../dataset")]
        root = next((p for p in candidates if (p / "train/_annotations.coco.json").exists()), None)
        if root is None:
            raise FileNotFoundError("Isi RAW_DATASET dengan folder dataset.")
    return root.resolve()

RAW = locate_dataset()
raw_coco = {s: read_json(RAW / s / "_annotations.coco.json") for s in ["train", "valid", "test"]}
print("Dataset:", RAW)


### 2.2 Class mapping

Tetapkan kelas dari JSON train dan pisahkan kategori induk yang tidak memiliki anotasi.


In [ ]:
register_cell('cell-026', 'categories')
train_categories = raw_coco["train"]["categories"]
annotated_ids = {a["category_id"] for a in raw_coco["train"]["annotations"]}
parent_names = {c.get("supercategory") for c in train_categories}
categories = sorted([
    c for c in train_categories
    if not (c["name"] in parent_names and c["id"] not in annotated_ids
            and c.get("supercategory") != c["name"])
], key=lambda c: c["id"])
for split, data in raw_coco.items():
    assert data["categories"] == train_categories, f"Daftar kelas berbeda: {split}"
CLASS_IDS = [c["id"] for c in categories]
CLASS_NAMES = {c["id"]: c["name"] for c in categories}
SOURCE_ID = fingerprint(raw_coco)
print("Kelas segmentasi:", len(categories))


### 2.3 Fungsi dataset audit

Audit file gambar, ukuran, poligon, referensi anotasi, dan duplikasi gambar identik.


In [ ]:
register_cell('cell-028', 'audit_function')
def audit_split(split, data, hashes):
    images = {i["id"]: i for i in data["images"]}
    assert len(images) == len(data["images"]), "ID gambar ganda"
    assert len({a["id"] for a in data["annotations"]}) == len(data["annotations"])
    problems, annotated = [], Counter()
    for item in data["images"]:
        path = RAW / split / item["file_name"]
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        hashes[digest].append(f"{split}/{item['file_name']}")
        with Image.open(path) as image:
            actual = image.size
            image.verify()
        if actual != (item["width"], item["height"]):
            problems.append({"split": split, "file_name": item["file_name"], "actual": actual})
    for ann in data["annotations"]:
        assert ann["image_id"] in images and ann["category_id"] in CLASS_IDS
        assert isinstance(ann["segmentation"], list) and ann["segmentation"]
        item = images[ann["image_id"]]
        for polygon in ann["segmentation"]:
            xy = np.asarray(polygon).reshape(-1, 2)
            assert len(xy) >= 3 and np.isfinite(xy).all()
            assert (xy >= 0).all() and (xy <= [item["width"], item["height"]]).all()
        annotated[ann["image_id"]] += 1
    return problems, len(set(images) - set(annotated))


### 2.4 Pemeriksaan dataset split

Tampilkan hasil audit dan simpan distribusi kelas untuk laporan.


In [ ]:
register_cell('cell-030', 'audit_run')
hashes = defaultdict(list)
audit_rows, size_issues, distribution = [], [], []
for split, data in raw_coco.items():
    issues, empty = audit_split(split, data, hashes)
    size_issues.extend(issues)
    audit_rows.append({"split": split, "images": len(data["images"]),
                       "annotations": len(data["annotations"]), "empty_images": empty})
    for category in categories:
        anns = [a for a in data["annotations"] if a["category_id"] == category["id"]]
        distribution.append({"split": split, "category_id": category["id"],
                             "name": category["name"], "instances": len(anns),
                             "images": len({a["image_id"] for a in anns})})
distribution = pd.DataFrame(distribution)
duplicates = [paths for paths in hashes.values() if len(paths) > 1]
SOURCE_ID = fingerprint({"annotations": raw_coco, "image_sha256": sorted(hashes)})
write_json(REPORTS / "audit.json", {"source_id": SOURCE_ID, "splits": audit_rows,
           "size_issues": size_issues, "exact_duplicates": duplicates})
distribution.to_csv(REPORTS / "class_distribution.csv", index=False)
display(pd.DataFrame(audit_rows))
display(pd.DataFrame(size_issues))
print("Kelompok duplikasi identik:", len(duplicates))
if any(len({p.split('/')[0] for p in group}) > 1 for group in duplicates):
    raise ValueError("Duplikasi lintas split perlu ditinjau sebelum training.")


### 2.5 Distribusi instance per class

Lihat ketimpangan distribusi kelas berdasarkan jumlah instance pada train.


In [ ]:
register_cell('cell-032', 'distribution_plot')
train_distribution = distribution.query("split == 'train'").set_index("name")
train_distribution["instances"].sort_values().plot.barh(figsize=(9, 16))
plt.xlabel("Jumlah instance train")
plt.tight_layout()
plt.savefig(REPORTS / "train_distribution.png", dpi=120)
plt.show()
display(train_distribution.query("instances == 0"))


## 3. Normalisasi data dan pemeriksaan mask


### 3.1 Konfigurasi orientation correction

Daftarkan koreksi orientasi empat gambar v5 yang ukuran pikselnya tertukar terhadap anotasi. Periksa overlay sebelum training.


In [ ]:
register_cell('cell-035', 'normalization_helpers')
ROTATE_CLOCKWISE = {
    "182169107_jpg.rf.c9b1bbecf1474a5b36a6df197c30dfed.jpg",
    "182169121_jpg.rf.eb0ccb71d0ea38adf1e38bf9c02a4a15.jpg",
    "182169109_jpg.rf.0531e60eb6e404d9633ed0c23a381fe7.jpg",
    "182169108_jpg.rf.205d9f1ed8f888006b8f97fc29eb8790.jpg",
}
unexpected = {row["file_name"] for row in size_issues} - ROTATE_CLOCKWISE
assert not unexpected, f"Masalah ukuran baru perlu diperiksa: {unexpected}"
corrections = []

def polygon_rle(segmentation, height, width):
    return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))

def mask_from_annotation(ann, image):
    rle = polygon_rle(ann["segmentation"], image["height"], image["width"])
    return mask_utils.decode(rle).astype(np.uint8)


### 3.2 Penyalinan image

Buat salinan gambar yang ukurannya konsisten dengan koordinat anotasi.


In [ ]:
register_cell('cell-037', 'copy_images')
def copy_images(source_folder, output_folder, data, apply_rotation=False):
    output_folder.mkdir(parents=True, exist_ok=True)
    for item in data["images"]:
        name = item["file_name"]
        assert Path(name).name == name, "Nama file gambar harus tanpa subdirektori."
        source, target = source_folder / name, output_folder / name
        with Image.open(source) as image:
            mismatch = image.size != (item["width"], item["height"])
            if mismatch:
                assert apply_rotation and name in ROTATE_CLOCKWISE, name
                fixed = image.transpose(Image.Transpose.ROTATE_270)
                assert fixed.size == (item["width"], item["height"])
                fixed.convert("RGB").save(target, quality=100, subsampling=0)
                corrections.append({"file_name": name, "action": "rotate_90_clockwise"})
            elif not target.exists():
                shutil.copy2(source, target)


### 3.3 Normalisasi COCO annotation

Hitung luas mask dan bounding box dari segmentasi untuk salinan JSON yang dipakai evaluator.


In [ ]:
register_cell('cell-039', 'normalize_annotations')
def normalized_annotations(data, keep_ids):
    result = copy.deepcopy(data)
    result["categories"] = [copy.deepcopy(c) for c in categories if c["id"] in keep_ids]
    images = {i["id"]: i for i in data["images"]}
    result["annotations"] = []
    for ann in data["annotations"]:
        if ann["category_id"] not in keep_ids:
            continue
        item = images[ann["image_id"]]
        rle = polygon_rle(ann["segmentation"], item["height"], item["width"])
        assert mask_utils.area(rle) > 0, f"Mask kosong: {ann['id']}"
        row = copy.deepcopy(ann)
        row["id"] = len(result["annotations"]) + 1
        row["area"] = float(mask_utils.area(rle))
        row["bbox"] = mask_utils.toBbox(rle).tolist()
        row["iscrowd"] = int(ann.get("iscrowd", 0))
        result["annotations"].append(row)
    return result


### 3.4 Dataset preparation

Normalisasi seluruh split dan rekam perubahan tanpa mengganti file sumber.


In [ ]:
register_cell('cell-041', 'prepare_full')
FULL_COCO = PREPARED / "coco_all"
full_coco = {}
for split, data in raw_coco.items():
    folder = FULL_COCO / split
    copy_images(RAW / split, folder, data, apply_rotation=True)
    full_coco[split] = normalized_annotations(data, CLASS_IDS)
    write_json(folder / "_annotations.coco.json", full_coco[split])
DATASET_ID = fingerprint({"source_id": SOURCE_ID, "normalized": full_coco})
if "EXPECTED_DATASET_ID" in globals():
    assert DATASET_ID == EXPECTED_DATASET_ID, "Dataset fingerprint worker berbeda dari parent."
write_json(REPORTS / "normalization.json", {
    "source_id": SOURCE_ID, "dataset_id": DATASET_ID,
    "corrections": corrections, "area": "rasterized_segmentation",
    "bbox": "rasterized_segmentation", "class_ids": CLASS_IDS,
})
print("Identitas dataset:", DATASET_ID)


### 3.5 Pemeriksaan ground-truth overlay

Tampilkan gambar dan mask asli yang sudah diselaraskan untuk pemeriksaan visual.


In [ ]:
register_cell('cell-043', 'overlay')
def show_ground_truth(split, file_name, axis=None):
    data = full_coco[split]
    item = next(i for i in data["images"] if i["file_name"] == file_name)
    axis = axis or plt.subplots(figsize=(8, 6))[1]
    axis.imshow(Image.open(FULL_COCO / split / file_name))
    for ann in data["annotations"]:
        if ann["image_id"] != item["id"]:
            continue
        for polygon in ann["segmentation"]:
            xy = np.asarray(polygon).reshape(-1, 2)
            axis.fill(xy[:, 0], xy[:, 1], alpha=0.2)
            axis.plot(*np.vstack([xy, xy[0]]).T, linewidth=1)
    axis.set_title(f"{split}: {file_name[:22]}")
    axis.axis("off")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for axis, row in zip(axes.flat, size_issues):
    show_ground_truth(row["split"], row["file_name"], axis)
plt.tight_layout()
plt.savefig(REPORTS / "orientation_review.png", dpi=130)
plt.show()


## 4. Skenario kelas dan format YOLO


### 4.1 Pemilihan class scenario

Cari hasil run sebelumnya dan daftar tiga kelas tersulit yang sudah ditetapkan.


In [ ]:
register_cell('cell-046', 'scenario')
search_roots = [Path(p) for p in PREVIOUS_RESULTS]
if IS_KAGGLE and not PREVIOUS_RESULTS:
    search_roots.append(Path("/kaggle/input"))
search_roots.append(WORK)

def find_artifacts(name):
    paths = []
    for root in search_roots:
        if root.exists():
            paths.extend(root.rglob(name))
    return sorted(set(p.resolve() for p in paths))

def load_hard3():
    paths = [Path(HARD3_FILE)] if HARD3_FILE else find_artifacts("hard3_classes.json")
    records = [read_json(p) for p in paths]
    records = [r for r in records if r.get("dataset_id") == DATASET_ID]
    choices = {tuple(r["category_ids"]) for r in records}
    assert len(choices) <= 1, "Daftar hard3 berbeda; isi HARD3_FILE secara eksplisit."
    return records[0] if records else None

hard3_record = load_hard3()
ACTIVE_IDS = CLASS_IDS if SCENARIO == "all_classes" else (hard3_record or {}).get("category_ids", [])
if SCENARIO == "hard3":
    assert len(ACTIVE_IDS) == 3, "Jalankan select_classes setelah tiga baseline tersedia."
ID_TO_INDEX = {cid: index for index, cid in enumerate(ACTIVE_IDS)}
INDEX_TO_ID = {index: cid for cid, index in ID_TO_INDEX.items()}
print("Skenario:", SCENARIO, "Jumlah kelas:", len(ACTIVE_IDS))


### 4.2 Dataset per scenario

Buat COCO skenario terpilih dengan daftar kelas yang konsisten pada semua split.


In [ ]:
register_cell('cell-048', 'scenario_data')
SCENARIO_COCO = PREPARED / f"coco_{SCENARIO}"
scenario_coco = {}
for split, data in full_coco.items():
    scenario_coco[split] = normalized_annotations(data, ACTIVE_IDS)
    destination = SCENARIO_COCO / split
    copy_images(FULL_COCO / split, destination, data)
    write_json(destination / "_annotations.coco.json", scenario_coco[split])
mapping = {"dataset_id": DATASET_ID, "scenario": SCENARIO,
           "category_ids": ACTIVE_IDS, "names": [CLASS_NAMES[c] for c in ACTIVE_IDS],
           "yolo_and_rfdetr": ID_TO_INDEX,
           "maskrcnn": {cid: index + 1 for cid, index in ID_TO_INDEX.items()}}
write_json(PREPARED / f"mapping_{SCENARIO}.json", mapping)


### 4.3 Konversi YOLO Segmentation

Konversi poligon COCO menjadi label YOLO Segmentation tanpa mengubah pembagian data.


In [ ]:
register_cell('cell-050', 'yolo_conversion')
YOLO_DATA = PREPARED / f"yolo_{SCENARIO}"
for split, data in scenario_coco.items():
    image_dir, label_dir = YOLO_DATA / "images" / split, YOLO_DATA / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    by_image = defaultdict(list)
    for ann in data["annotations"]:
        by_image[ann["image_id"]].append(ann)
    for item in data["images"]:
        source, target = SCENARIO_COCO / split / item["file_name"], image_dir / item["file_name"]
        if not target.exists():
            shutil.copy2(source, target)
        lines = []
        for ann in by_image[item["id"]]:
            assert len(ann["segmentation"]) == 1, "Konversi multi-poligon memerlukan penanganan khusus."
            xy = np.asarray(ann["segmentation"][0]).reshape(-1, 2)
            xy = xy / [item["width"], item["height"]]
            values = " ".join(f"{value:.8f}" for value in xy.ravel())
            lines.append(f"{ID_TO_INDEX[ann['category_id']]} {values}")
        (label_dir / f"{Path(item['file_name']).stem}.txt").write_text("\n".join(lines), encoding="utf-8")
yolo_config = {"path": str(YOLO_DATA.resolve()), "train": "images/train", "val": "images/valid",
               "test": "images/test", "names": dict(enumerate(mapping["names"]))}
(YOLO_DATA / "data.yaml").write_text(yaml.safe_dump(yolo_config, sort_keys=False), encoding="utf-8")


## 5. Evaluator dan format prediksi bersama


### 5.1 Format prediction COCO RLE

Simpan mask prediksi sebagai COCO RLE beserta skor dan ID kelas asli.


In [ ]:
register_cell('cell-053', 'prediction_format')
def prediction_record(image_id, category_id, score, binary_mask):
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return {"image_id": int(image_id), "category_id": int(category_id),
            "score": float(score), "segmentation": rle}

def write_predictions(path, predictions):
    with gzip.open(path, "wt", encoding="utf-8") as handle:
        json.dump(predictions, handle)

def read_predictions(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return json.load(handle)

def make_coco(data):
    coco = COCO()
    coco.dataset = copy.deepcopy(data)
    with redirect_stdout(io.StringIO()):
        coco.createIndex()
    return coco


### 5.2 COCO mask evaluator

Hitung mask AP dengan COCOeval yang sama untuk seluruh model, termasuk ketika tidak ada prediksi.


In [ ]:
register_cell('cell-055', 'evaluator')
def evaluate_predictions(data, predictions, category_ids=None):
    gt = make_coco(data)
    with redirect_stdout(io.StringIO()):
        dt = gt.loadRes(copy.deepcopy(predictions)) if predictions else make_coco({
            "images": data["images"], "categories": data["categories"], "annotations": []})
        evaluator = COCOeval(gt, dt, "segm")
        evaluator.params.catIds = sorted(category_ids or [c["id"] for c in data["categories"]])
        evaluator.params.maxDets = [1, 10, MAX_DETECTIONS]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    rows = []
    precision = evaluator.eval["precision"]
    for index, cid in enumerate(evaluator.params.catIds):
        values = precision[:, :, index, 0, -1]
        valid = values[values >= 0]
        values50 = precision[0, :, index, 0, -1]
        rows.append({"category_id": cid, "name": CLASS_NAMES[cid],
                     "gt_instances": sum(a["category_id"] == cid for a in data["annotations"]),
                     "gt_images": len({a["image_id"] for a in data["annotations"] if a["category_id"] == cid}),
                     "mask_ap": float(valid.mean()) if len(valid) else None,
                     "mask_ap50": float(values50[values50 >= 0].mean()) if (values50 >= 0).any() else None})
    summary = {"mask_ap": float(evaluator.stats[0]), "mask_ap50": float(evaluator.stats[1]),
               "mask_ap75": float(evaluator.stats[2]), "mask_ar100": float(evaluator.stats[8])}
    return summary, pd.DataFrame(rows)


## 6. Direktori run dan pencatatan proses


### 6.1 Run metadata

Pisahkan hasil setiap model, skenario, dan label run. Mode prediksi memakai checkpoint dari run training yang sudah ada.


In [ ]:
register_cell('cell-058', 'run_config')
MODEL_NAMES = {"yolo11": "YOLO11n-seg", "maskrcnn": "Mask R-CNN ResNet-50 FPN",
               "rfdetr": "RF-DETR Seg Nano"}
run_name = f"{RUN_LABEL}_smoke" if SMOKE_TEST else RUN_LABEL
RUN_DIR = (WORK / "sequence" / SCENARIO / run_name if MODEL == "all"
           else WORK / "artifacts" / MODEL / SCENARIO / run_name)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG = {"nrp": "5054241018", "model": MODEL, "model_name": MODEL_NAMES.get(MODEL, "Sequential training"),
              "scenario": SCENARIO, "run_label": run_name, "seed": SEED,
              "epochs": 1 if SMOKE_TEST else EPOCHS, "smoke_test": SMOKE_TEST,
              "dataset_id": DATASET_ID, "category_ids": ACTIVE_IDS,
              "resolution": {"yolo11": YOLO_SIZE, "maskrcnn": [MASKRCNN_SHORT, MASKRCNN_LONG], "rfdetr": RFDETR_SIZE},
              "batch_size": {"yolo11": YOLO_BATCH, "maskrcnn": MASKRCNN_BATCH, "rfdetr": RFDETR_BATCH},
              "score_floor": SCORE_FLOOR, "max_detections": MAX_DETECTIONS}
if MODE == "train":
    previous = RUN_DIR / "run_config.json"
    if previous.exists():
        assert read_json(previous) == RUN_CONFIG, "Konfigurasi berubah; gunakan RUN_LABEL baru."
    write_json(previous, RUN_CONFIG)
print("Output:", RUN_DIR)


### 6.2 Konfigurasi PyTorch dan device

Siapkan PyTorch hanya ketika model akan dijalankan dan catat lingkungan eksekusinya.


In [ ]:
register_cell('cell-060', 'torch_setup')
DEVICE = "cpu"
if needs_model:
    import torch
    import torchvision
    import torchvision.transforms.functional as TF
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        DEVICE = "cuda"
    if MODE == "train" and not SMOKE_TEST and DEVICE != "cuda":
        raise RuntimeError("Aktifkan GPU Kaggle sebelum training utama.")
    versions = {}
    for name in ["torch", "torchvision", "ultralytics", "rfdetr", "pycocotools", "numpy"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    environment = {"python": sys.version, "versions": versions, "device": DEVICE,
                   "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
    write_json(RUN_DIR / f"environment_{MODE}.json", environment)
    print(environment)


### 6.3 Training duration dan recovery

Catat durasi dan status training meskipun proses mengalami kesalahan.


In [ ]:
register_cell('cell-062', 'timing')
def timed_training(function):
    started, stamp = time.perf_counter(), datetime.now(timezone.utc).isoformat()
    status = "failed"
    try:
        result = function()
        status = "finished"
        return result
    except KeyboardInterrupt:
        status = "interrupted"
        raise
    finally:
        record = {"started_utc": stamp, "seconds": time.perf_counter() - started,
                  "status": status, "model": MODEL, "scenario": SCENARIO,
                  "device": DEVICE, "smoke_test": SMOKE_TEST}
        try:
            path = RUN_DIR / "training_sessions.json"
            sessions = read_json(path) if path.exists() else []
            write_json(path, sessions + [record])
        except Exception as error:
            print("Catatan durasi belum tersimpan:", error)
        try:
            export_recovery(RUN_DIR)
        except Exception as error:
            print("Ekspor otomatis gagal; checkpoint yang sudah ditulis tetap perlu diperiksa:", error)

predict_one = None
CHECKPOINT = None


### 6.4 Atomic checkpoint write

Simpan checkpoint melalui file sementara sebelum mengganti salinan terakhir yang lengkap.


In [ ]:
register_cell('cell-064', 'atomic_checkpoints')
def atomic_copy(source, destination):
    destination = Path(destination)
    partial = destination.with_name(destination.name + ".tmp")
    try:
        shutil.copy2(source, partial)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()

def atomic_torch_save(state, destination):
    destination = Path(destination)
    partial = destination.with_name(destination.name + ".tmp")
    try:
        torch.save(state, partial)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()


### 6.5 Smoke-test subset

Pisahkan data uji singkat dari data eksperimen utama.


In [ ]:
register_cell('cell-066', 'smoke_data')
TRAIN_ROOT = SCENARIO_COCO
YOLO_YAML = YOLO_DATA / "data.yaml"
if SMOKE_TEST:
    TRAIN_ROOT = PREPARED / f"smoke_{SCENARIO}"
    for split, limit in [("train", 12), ("valid", 4), ("test", 4)]:
        data = copy.deepcopy(scenario_coco[split])
        data["images"] = data["images"][:limit]
        ids = {i["id"] for i in data["images"]}
        data["annotations"] = [a for a in data["annotations"] if a["image_id"] in ids]
        copy_images(SCENARIO_COCO / split, TRAIN_ROOT / split, data)
        write_json(TRAIN_ROOT / split / "_annotations.coco.json", data)
    smoke_yaml = copy.deepcopy(yolo_config)
    for split, key in [("train", "train"), ("valid", "val")]:
        items = read_json(TRAIN_ROOT / split / "_annotations.coco.json")["images"]
        listing = YOLO_DATA / f"smoke_{split}.txt"
        listing.write_text("\n".join(str((YOLO_DATA / "images" / split / i["file_name"]).resolve())
                                      for i in items), encoding="utf-8")
        smoke_yaml[key] = str(listing.resolve())
    YOLO_YAML = YOLO_DATA / "data_smoke.yaml"
    YOLO_YAML.write_text(yaml.safe_dump(smoke_yaml, sort_keys=False), encoding="utf-8")


### 6.6 Checkpoint compatibility

Periksa identitas checkpoint sebelum melanjutkan training atau menjalankan inference.


In [ ]:
register_cell('cell-068', 'checkpoint_guard')
def verify_checkpoint(path):
    path = Path(path)
    assert path.is_file(), f"Checkpoint tidak ditemukan: {path}"
    config_path = path.parent / "run_config.json"
    assert config_path.exists(), "Letakkan run_config.json di samping checkpoint."
    config = read_json(config_path)
    for key in ["model", "scenario", "dataset_id", "category_ids", "smoke_test", "resolution"]:
        assert config[key] == RUN_CONFIG[key], f"Checkpoint berbeda pada {key}"
    return path

if RESUME_CHECKPOINT:
    assert MODE == "train"
    verify_checkpoint(RESUME_CHECKPOINT)
    source_folder = Path(RESUME_CHECKPOINT).parent
    if MODEL == "rfdetr" and source_folder.name == "native":
        source_folder = source_folder.parent
    source_config = read_json(source_folder / "run_config.json")
    assert source_config["run_label"] == run_name, "Resume memakai RUN_LABEL yang sama dengan checkpoint."
    for name in ["best.pt", "best.pth", "best_validation.json", "training_sessions.json"]:
        if (source_folder / name).exists() and (source_folder / name).resolve() != (RUN_DIR / name).resolve():
            shutil.copy2(source_folder / name, RUN_DIR / name)
    if MODEL == "rfdetr":
        native_source = source_folder if source_folder.name == "native" else source_folder / "native"
        if native_source.exists() and native_source.resolve() != (RUN_DIR / "native").resolve():
            shutil.copytree(native_source, RUN_DIR / "native", dirs_exist_ok=True)
if MODE in {"predict", "diagnose"}:
    assert INFERENCE_CHECKPOINT, "Isi INFERENCE_CHECKPOINT dengan checkpoint terbaik."
    CHECKPOINT = verify_checkpoint(INFERENCE_CHECKPOINT)
    trained_config = read_json(CHECKPOINT.parent / "run_config.json")
    RUN_CONFIG = {**trained_config, "run_label": run_name,
                  "score_floor": SCORE_FLOOR, "max_detections": MAX_DETECTIONS}
    write_json(RUN_DIR / "run_config.json", RUN_CONFIG)
if MODE == "train" and not RESUME_CHECKPOINT:
    assert not (RUN_DIR / "best.pt").exists() and not (RUN_DIR / "best.pth").exists(), "Run sudah ada; gunakan RUN_LABEL baru atau resume."


## 7. YOLO11n-seg


### 7.1 Inisialisasi YOLO11n-seg

Muat YOLO11 untuk model terpilih. Section model lain dilewati secara otomatis.


In [ ]:
register_cell('cell-071', 'yolo_model')
if needs_model and MODEL == "yolo11":
    from ultralytics import YOLO
    yolo_source = RESUME_CHECKPOINT or (str(CHECKPOINT) if CHECKPOINT else "yolo11n-seg.pt")
    yolo_model = YOLO(yolo_source)
    yolo_best_ap = -1.0
    if RESUME_CHECKPOINT:
        score_file = Path(RESUME_CHECKPOINT).parent / "best_validation.json"
        if score_file.exists():
            yolo_best_ap = read_json(score_file)["mask_ap"]


### 7.2 YOLO checkpoint callback

Simpan checkpoint YOLO berdasarkan mask AP validation dan pertahankan checkpoint terakhir sebelum optimizer dibuang.


In [ ]:
register_cell('cell-073', 'yolo_checkpoint')
def save_yolo_epoch(trainer):
    global yolo_best_ap
    score = trainer.metrics.get("metrics/mAP50-95(M)")
    if not Path(trainer.last).exists():
        return
    atomic_copy(trainer.last, RUN_DIR / "last.pt")
    if score is not None and float(score) > yolo_best_ap:
        yolo_best_ap = float(score)
        atomic_copy(trainer.last, RUN_DIR / "best.pt")
        write_json(RUN_DIR / "best_validation.json", {
            "mask_ap": yolo_best_ap, "epoch": int(trainer.epoch + 1), "selection": "native_validation_mask_ap"})

if needs_model and MODEL == "yolo11":
    yolo_model.add_callback("on_model_save", save_yolo_epoch)


### 7.3 Training YOLO11n-seg

Latih YOLO11 pada skenario terpilih. Data test tidak dipakai memilih checkpoint.


In [ ]:
register_cell('cell-075', 'yolo_train')
def train_yolo():
    yolo_model.train(
        data=str(YOLO_YAML), epochs=RUN_CONFIG["epochs"], imgsz=YOLO_SIZE,
        batch=YOLO_BATCH, device=0 if DEVICE == "cuda" else "cpu",
        workers=2 if IS_KAGGLE else 0, seed=SEED, deterministic=True,
        project=str(RUN_DIR), name="native", exist_ok=True,
        resume=bool(RESUME_CHECKPOINT), patience=EPOCHS + 1, save=True,
        plots=True, amp=DEVICE == "cuda", overlap_mask=False,
    )
    native_csv = Path(yolo_model.trainer.save_dir) / "results.csv"
    if native_csv.exists():
        shutil.copy2(native_csv, RUN_DIR / "training_log.csv")
    assert (RUN_DIR / "best.pt").exists(), "Mask AP tidak ditemukan pada callback YOLO."
    return RUN_DIR / "best.pt"

if MODE == "train" and MODEL == "yolo11":
    CHECKPOINT = timed_training(train_yolo)


### 7.4 YOLO inference adapter

Ubah keluaran YOLO menjadi kelas asli dan mask berukuran gambar sumber.


In [ ]:
register_cell('cell-077', 'yolo_predictor')
def predict_yolo(image):
    result = yolo_model.predict(image, imgsz=YOLO_SIZE, conf=SCORE_FLOOR,
                                max_det=MAX_DETECTIONS, retina_masks=True,
                                device=0 if DEVICE == "cuda" else "cpu", verbose=False)[0]
    if result.masks is None:
        return []
    rows = []
    for label, score, mask in zip(result.boxes.cls.cpu().numpy(), result.boxes.conf.cpu().numpy(),
                                   result.masks.data.cpu().numpy()):
        binary = Image.fromarray((mask > 0.5).astype(np.uint8))
        binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST))
        rows.append((INDEX_TO_ID[int(label)], float(score), binary))
    return rows

if needs_model and MODEL == "yolo11":
    if MODE == "train":
        yolo_model = YOLO(str(CHECKPOINT))
    predict_one = predict_yolo


## 8. Mask R-CNN ResNet-50 FPN


### 8.1 PyTorch COCO Dataset

Buat dataset PyTorch yang mengubah poligon menjadi mask biner dan menyediakan label background terpisah.


In [ ]:
register_cell('cell-080', 'mask_dataset')
if needs_model:
    class IndoorCocoDataset(torch.utils.data.Dataset):
        def __init__(self, folder, augment=False):
            self.folder = Path(folder)
            self.data = read_json(self.folder / "_annotations.coco.json")
            self.images = self.data["images"]
            self.by_image = defaultdict(list)
            self.augment = augment
            for ann in self.data["annotations"]:
                self.by_image[ann["image_id"]].append(ann)

        def __len__(self):
            return len(self.images)

        def __getitem__(self, index):
            item = self.images[index]
            image = Image.open(self.folder / item["file_name"]).convert("RGB")
            anns = self.by_image[item["id"]]
            masks = np.stack([mask_from_annotation(a, item) for a in anns]) if anns else np.zeros((0, item["height"], item["width"]), dtype=np.uint8)
            boxes = np.asarray([a["bbox"] for a in anns], dtype=np.float32).reshape(-1, 4)
            boxes[:, 2:] += boxes[:, :2]
            target = {"boxes": torch.as_tensor(boxes), "masks": torch.as_tensor(masks),
                      "labels": torch.tensor([ID_TO_INDEX[a["category_id"]] + 1 for a in anns], dtype=torch.int64),
                      "image_id": torch.tensor(item["id"]),
                      "area": torch.tensor([a["area"] for a in anns], dtype=torch.float32),
                      "iscrowd": torch.tensor([a.get("iscrowd", 0) for a in anns], dtype=torch.int64)}
            image = TF.to_tensor(image)
            if self.augment and random.random() < 0.5:
                image = image.flip(-1)
                target["masks"] = target["masks"].flip(-1)
                target["boxes"][:, [0, 2]] = item["width"] - target["boxes"][:, [2, 0]]
            return image, target


### 8.2 Arsitektur Mask R-CNN

Bangun Mask R-CNN dan ganti head klasifikasi serta mask sesuai jumlah kelas aktif.


In [ ]:
register_cell('cell-082', 'mask_model')
def build_maskrcnn(pretrained=True):
    from torchvision.models.detection import MaskRCNN, maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
    from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    from torchvision.ops import FrozenBatchNorm2d
    options = dict(min_size=MASKRCNN_SHORT, max_size=MASKRCNN_LONG,
                   box_score_thresh=SCORE_FLOOR, box_detections_per_img=MAX_DETECTIONS)
    if pretrained:
        model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,
                                     weights_backbone=None, **options)
    else:
        backbone = resnet_fpn_backbone(backbone_name="resnet50", weights=None,
                                       norm_layer=FrozenBatchNorm2d, trainable_layers=3)
        model = MaskRCNN(backbone, num_classes=91, **options)
        for layer in model.modules():
            if isinstance(layer, FrozenBatchNorm2d):
                layer.eps = 0.0
    features = model.roi_heads.box_predictor.cls_score.in_features
    channels = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.box_predictor = FastRCNNPredictor(features, len(ACTIVE_IDS) + 1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(channels, 256, len(ACTIVE_IDS) + 1)
    return model.to(DEVICE)

if needs_model and MODEL == "maskrcnn":
    mask_model = build_maskrcnn(pretrained=MODE == "train" and not RESUME_CHECKPOINT)


### 8.3 Training step Mask R-CNN

Jalankan satu epoch Mask R-CNN dan hentikan proses jika loss tidak valid.


In [ ]:
register_cell('cell-084', 'mask_epoch')
def train_mask_epoch(loader, optimizer, scaler):
    mask_model.train()
    totals, batches = Counter(), 0
    for images, targets in loader:
        images = [image.to(DEVICE) for image in images]
        targets = [{key: value.to(DEVICE) for key, value in target.items()} for target in targets]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
            losses = mask_model(images, targets)
            loss = sum(losses.values())
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Loss tidak valid: {losses}")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        totals.update({key: float(value.detach()) for key, value in losses.items()})
        batches += 1
    return {key: value / max(batches, 1) for key, value in totals.items()}


### 8.4 Optimizer dan resume state

Siapkan optimizer, scheduler, dan pemulihan checkpoint Mask R-CNN.


In [ ]:
register_cell('cell-086', 'mask_optimizer')
def setup_mask_training():
    dataset = IndoorCocoDataset(TRAIN_ROOT / "train", augment=True)
    loader = torch.utils.data.DataLoader(dataset, batch_size=MASKRCNN_BATCH, shuffle=True,
                                          num_workers=0, collate_fn=lambda batch: tuple(zip(*batch)))
    optimizer = torch.optim.SGD([p for p in mask_model.parameters() if p.requires_grad],
                                lr=0.005, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    start_epoch, best_ap, history = 0, -1.0, []
    if RESUME_CHECKPOINT:
        state = torch.load(RESUME_CHECKPOINT, map_location=DEVICE, weights_only=False)
        mask_model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best_ap, history = state["epoch"], state["best_ap"], state["history"]
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"].cpu())
        if DEVICE == "cuda" and state.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng"])
    return loader, optimizer, scheduler, scaler, start_epoch, best_ap, history


### 8.5 Mask R-CNN inference adapter

Ubah prediksi Mask R-CNN ke format mask dan ID kelas bersama.


In [ ]:
register_cell('cell-088', 'mask_predictor')
def predict_maskrcnn(image):
    mask_model.eval()
    with torch.inference_mode():
        result = mask_model([TF.to_tensor(image).to(DEVICE)])[0]
    rows = []
    for label, score, mask in zip(result["labels"].cpu().numpy(), result["scores"].cpu().numpy(),
                                   result["masks"][:, 0].cpu().numpy()):
        if score >= SCORE_FLOOR:
            rows.append((INDEX_TO_ID[int(label) - 1], float(score), (mask >= 0.5).astype(np.uint8)))
    return rows

def predict_dataset(folder, data, predictor, brightness=None):
    predictions = []
    for position, item in enumerate(data["images"]):
        with Image.open(Path(folder) / item["file_name"]) as source:
            image = source.convert("RGB")
        if brightness is not None:
            image = ImageEnhance.Brightness(image).enhance(brightness)
        for cid, score, mask in predictor(image):
            assert mask.shape == (item["height"], item["width"]), "Ukuran mask prediksi tidak sesuai."
            predictions.append(prediction_record(item["id"], cid, score, mask))
        if (position + 1) % 50 == 0:
            print(f"Inferensi {position + 1}/{len(data['images'])}")
    return predictions


### 8.6 Training Mask R-CNN

Latih Mask R-CNN, validation setiap epoch, dan simpan keadaan training untuk dilanjutkan.


In [ ]:
register_cell('cell-090', 'mask_train')
def train_maskrcnn():
    loader, optimizer, scheduler, scaler, start, best_ap, history = setup_mask_training()
    validation = read_json(TRAIN_ROOT / "valid/_annotations.coco.json")
    for epoch in range(start, RUN_CONFIG["epochs"]):
        started = time.perf_counter()
        losses = train_mask_epoch(loader, optimizer, scaler)
        predictions = predict_dataset(TRAIN_ROOT / "valid", validation, predict_maskrcnn)
        metrics, _ = evaluate_predictions(validation, predictions, ACTIVE_IDS)
        scheduler.step()
        improved = metrics["mask_ap"] > best_ap
        best_ap = max(best_ap, metrics["mask_ap"])
        history.append({"epoch": epoch + 1, **losses, **metrics, "seconds": time.perf_counter() - started})
        state = {"model": mask_model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                 "epoch": epoch + 1, "best_ap": best_ap, "history": history,
                 "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                 "torch_rng": torch.get_rng_state(),
                 "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}
        atomic_torch_save(state, RUN_DIR / "last.pth")
        if improved:
            atomic_torch_save({"model": state["model"], "epoch": epoch + 1}, RUN_DIR / "best.pth")
        pd.DataFrame(history).to_csv(RUN_DIR / "training_log.csv", index=False)
        print(history[-1])
    assert (RUN_DIR / "best.pth").exists(), "Tidak ada checkpoint terbaik pada run ini."
    return RUN_DIR / "best.pth"

if MODE == "train" and MODEL == "maskrcnn":
    CHECKPOINT = timed_training(train_maskrcnn)


### 8.7 Pemuatan best checkpoint

Muat checkpoint Mask R-CNN terbaik untuk ekspor prediksi.


In [ ]:
register_cell('cell-092', 'mask_load')
if needs_model and MODEL == "maskrcnn":
    state = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
    mask_model.load_state_dict(state["model"])
    mask_model.eval()
    predict_one = predict_maskrcnn


## 9. RF-DETR Seg Nano


### 9.1 Konfigurasi RF-DETR Seg Nano

Muat RF-DETR Seg Nano dan siapkan konfigurasi training dari COCO yang sama.


In [ ]:
register_cell('cell-095', 'rf_model_train')
if needs_model and MODEL == "rfdetr":
    from rfdetr import RFDETR, RFDETRSegNano
    if MODE == "train":
        rf_model = RFDETRSegNano(resolution=RFDETR_SIZE, device=DEVICE)
    else:
        rf_model = RFDETR.from_checkpoint(str(CHECKPOINT), device=DEVICE)

def train_rfdetr():
    write_json(RUN_DIR / "native/run_config.json", RUN_CONFIG)
    rf_model.train(
        dataset_dir=str(TRAIN_ROOT), output_dir=str(RUN_DIR / "native"),
        epochs=RUN_CONFIG["epochs"], batch_size=RFDETR_BATCH, grad_accum_steps=1,
        lr=1e-4, lr_encoder=1e-5, resolution=RFDETR_SIZE,
        device=DEVICE, devices=1, num_workers=2 if IS_KAGGLE else 0,
        seed=SEED, run_test=False, early_stopping=False,
        eval_max_dets=MAX_DETECTIONS, tensorboard=True, wandb=False,
        checkpoint_interval=10, resume=RESUME_CHECKPOINT or None,
    )
    best = RUN_DIR / "native/checkpoint_best_total.pth"
    assert best.exists(), "Checkpoint RF-DETR belum tersedia. Periksa log native."
    atomic_copy(best, RUN_DIR / "best.pth")
    last = RUN_DIR / "native/last.ckpt"
    if last.exists():
        atomic_copy(last, RUN_DIR / "last.ckpt")
    return RUN_DIR / "best.pth"


### 9.2 Training RF-DETR Seg Nano

Jalankan training RF-DETR hanya bila dipilih dan muat hasil terbaiknya.


In [ ]:
register_cell('cell-097', 'rf_train')
if MODE == "train" and MODEL == "rfdetr":
    CHECKPOINT = timed_training(train_rfdetr)
    rf_model = RFDETR.from_checkpoint(str(CHECKPOINT), device=DEVICE)
if needs_model and MODEL == "rfdetr":
    assert len(rf_model.class_names) == len(ACTIVE_IDS), "Jumlah kelas checkpoint berbeda."
    assert list(rf_model.class_names) == mapping["names"], "Urutan kelas RF-DETR berbeda."


### 9.3 RF-DETR inference adapter

Petakan prediksi RF-DETR dari indeks kelas berurutan ke ID kelas asli.


In [ ]:
register_cell('cell-099', 'rf_predictor')
def predict_rfdetr(image):
    detections = rf_model.predict(image, threshold=SCORE_FLOOR)
    if detections.mask is None:
        return []
    order = np.argsort(-detections.confidence)[:MAX_DETECTIONS]
    return [(INDEX_TO_ID[int(detections.class_id[j])], float(detections.confidence[j]),
             np.asarray(detections.mask[j], dtype=np.uint8)) for j in order]

if needs_model and MODEL == "rfdetr":
    predict_one = predict_rfdetr


## 10. Ekspor dan evaluasi validation


### 10.1 Validation prediction dan metrics

Ekspor prediksi validation checkpoint terpilih. Hasil uji singkat tidak dipakai memilih kelas tersulit.


In [ ]:
register_cell('cell-102', 'export_valid')
def save_evaluation(split, data, predictions, scope="active"):
    summary, per_class = evaluate_predictions(data, predictions, ACTIVE_IDS)
    write_predictions(RUN_DIR / f"predictions_{split}.json.gz", predictions)
    per_class.to_csv(RUN_DIR / f"per_class_{split}.csv", index=False)
    record = {**RUN_CONFIG, **summary, "split": split, "scope": scope,
              "images": len(data["images"]), "checkpoint": str(CHECKPOINT)}
    write_json(RUN_DIR / f"metrics_{split}.json", record)
    return record

if needs_model and MODE in {"train", "predict"}:
    assert predict_one is not None and CHECKPOINT is not None
    validation = read_json(TRAIN_ROOT / "valid/_annotations.coco.json")
    validation_predictions = predict_dataset(TRAIN_ROOT / "valid", validation, predict_one)
    display(pd.DataFrame([save_evaluation("valid", validation, validation_predictions)]))


## 11. Pemilihan tiga kelas tersulit — NRP 5054241018


### 11.1 Pengumpulan run metrics

Kumpulkan satu run validation setiap model. Tolak pencampuran hasil dari dataset atau konfigurasi evaluasi berbeda.


In [ ]:
register_cell('cell-105', 'collect_metrics')
def collect_metrics(split=None):
    rows = []
    pattern = f"metrics_{split}.json" if split else "metrics_*.json"
    for path in find_artifacts(pattern):
        record = read_json(path)
        if record.get("dataset_id") != DATASET_ID or record.get("smoke_test"):
            continue
        record["artifact_dir"] = str(path.parent)
        rows.append(record)
    unique = {}
    for record in rows:
        key = (record["model"], record["scenario"], record["split"], record["scope"], record["run_label"])
        if key in unique:
            assert unique[key]["mask_ap"] == record["mask_ap"], f"Output berbeda untuk run {key}"
        unique[key] = record
    return list(unique.values())

def baseline_runs():
    records = [r for r in collect_metrics("valid") if r["scenario"] == "all_classes"]
    selected = {}
    for model in MODEL_NAMES:
        choices = [r for r in records if r["model"] == model]
        assert len(choices) == 1, f"Diperlukan satu baseline {model}; temukan {len(choices)}. Batasi PREVIOUS_RESULTS."
        selected[model] = choices[0]
    return selected


### 11.2 Pemilihan tiga kelas tersulit

Pilih kelas dengan AP validation rata-rata terendah yang memiliki cukup contoh train dan validation.


In [ ]:
register_cell('cell-107', 'select_classes')
if MODE == "select_classes":
    baselines = baseline_runs()
    tables = []
    for model, record in baselines.items():
        assert record["max_detections"] == MAX_DETECTIONS and record["score_floor"] == SCORE_FLOOR
        table = pd.read_csv(Path(record["artifact_dir"]) / "per_class_valid.csv")
        tables.append(table[["category_id", "mask_ap"]].rename(columns={"mask_ap": model}))
    ranking = tables[0]
    for table in tables[1:]:
        ranking = ranking.merge(table, on="category_id", validate="one_to_one")
    support = distribution.pivot(index="category_id", columns="split", values="images")
    ranking = ranking.join(support[["train", "valid"]], on="category_id")
    ranking = ranking[(ranking["train"] >= MIN_TRAIN_IMAGES) & (ranking["valid"] >= MIN_VALID_IMAGES)]
    ranking = ranking.dropna(subset=list(MODEL_NAMES))
    ranking["mean_validation_ap"] = ranking[list(MODEL_NAMES)].mean(axis=1)
    ranking = ranking.sort_values(["mean_validation_ap", "category_id"])
    assert len(ranking) >= 3, "Kandidat kelas belum cukup; tinjau aturan dukungan sebelum eksperimen."
    chosen = sorted(ranking.head(3)["category_id"].astype(int).tolist())
    choice = {"dataset_id": DATASET_ID, "category_ids": chosen,
              "names": [CLASS_NAMES[cid] for cid in chosen], "source_split": "valid",
              "min_train_images": MIN_TRAIN_IMAGES, "min_valid_images": MIN_VALID_IMAGES,
              "baseline_runs": {m: r["run_label"] for m, r in baselines.items()}}
    existing = load_hard3()
    assert existing is None or existing == choice, "Pilihan hard3 sudah terkunci dan berbeda."
    write_json(REPORTS / "hard3_classes.json", choice)
    ranking["name"] = ranking["category_id"].map(CLASS_NAMES)
    ranking.to_csv(REPORTS / "hard3_ranking.csv", index=False)
    display(ranking.head(10))


## 12. Test set foto HP sendiri


Bagian ini untuk tahap berikutnya dan dilewati saat `ENABLE_PERSONAL = False`.
Aktifkan flag tersebut dan set `PERSONAL_DATASET` ketika anotasi sudah tersedia sebagai
COCO Segmentation. Folder dapat langsung berisi gambar dan JSON, atau memiliki
subfolder `test`. Nama kelas harus mengikuti dataset v5.

Sediakan `photo_metadata.csv` dengan kolom `file_name`, `is_difficult`,
`condition`, dan `exif_proof`. `exif_proof` menunjuk file bukti metadata relatif
terhadap folder foto, misalnya `evidence/IMG_001.json`. Simpan screenshot proses
anotasi dalam folder `annotation_screenshots`. Notebook memeriksa kelengkapan
file, bukan membuktikan keaslian foto. Foto sendiri tidak masuk training.


### 12.1 Dataset foto mandiri

Baca anotasi foto sendiri dan cocokkan nama kelasnya dengan kelas v5.


In [ ]:
register_cell('cell-111', 'personal_load')
def load_personal():
    if not ENABLE_PERSONAL:
        return None
    assert PERSONAL_DATASET, "Isi PERSONAL_DATASET setelah foto dan anotasi tersedia."
    folder = Path(PERSONAL_DATASET)
    if not (folder / "_annotations.coco.json").exists():
        folder = folder / "test"
    data = read_json(folder / "_annotations.coco.json")
    assert len(data["images"]) >= 40, "Test sendiri wajib minimal 40 foto."
    names_to_id = {c["name"]: c["id"] for c in categories}
    own_categories = {c["id"]: c["name"] for c in data["categories"]}
    for ann in data["annotations"]:
        name = own_categories[ann["category_id"]]
        assert name in names_to_id, f"Kelas foto sendiri tidak dikenal: {name}"
        ann["category_id"] = names_to_id[name]
    assert len({i["id"] for i in data["images"]}) == len(data["images"])
    for item in data["images"]:
        assert Path(item["file_name"]).name == item["file_name"]
        with Image.open(folder / item["file_name"]) as image:
            assert image.size == (item["width"], item["height"]), item["file_name"]
    normalized = normalized_annotations(data, CLASS_IDS)
    destination = PREPARED / "personal"
    copy_images(folder, destination, normalized)
    write_json(destination / "_annotations.coco.json", normalized)
    return folder, destination, normalized


### 12.2 Metadata foto mandiri

Periksa minimal sepuluh foto sulit beserta bukti metadata dan screenshot anotasinya.


In [ ]:
register_cell('cell-113', 'personal_evidence')
def check_personal_evidence(folder, data):
    path = Path(PERSONAL_METADATA) if PERSONAL_METADATA else folder / "photo_metadata.csv"
    metadata = pd.read_csv(path, keep_default_na=False)
    required = {"file_name", "is_difficult", "condition", "exif_proof"}
    assert required <= set(metadata), f"Kolom CSV wajib: {required}"
    assert metadata["file_name"].is_unique
    assert set(metadata["file_name"]) == {i["file_name"] for i in data["images"]}
    difficult = metadata["is_difficult"].astype(str).str.lower().isin(["1", "true", "yes", "ya"])
    assert difficult.sum() >= 10, "Minimal 10 foto harus dalam kondisi sulit."
    assert metadata.loc[difficult, "condition"].str.strip().ne("").all()
    for proof in metadata["exif_proof"]:
        assert proof and (folder / proof).is_file(), f"Bukti metadata belum ada: {proof}"
    screenshots = folder / "annotation_screenshots"
    assert screenshots.exists() and any(p.is_file() for p in screenshots.rglob("*")), "Screenshot anotasi belum tersedia."
    metadata.to_csv(REPORTS / "personal_photo_metadata.csv", index=False)
    return metadata

personal = load_personal()
if personal is not None:
    personal_metadata = check_personal_evidence(personal[0], personal[2])
    print("Foto sendiri:", len(personal[2]["images"]))
else:
    print("Bagian foto HP dilewati. Fokus tahap ini: dataset Roboflow dan tiga model.")


## 13. Evaluasi akhir dan perbandingan pada kelas yang sama


Aktifkan `FINAL_TEST` setelah konfigurasi model ditetapkan dari validation.
Test asli Roboflow pada `all_classes` dapat dijalankan sebelum eksperimen hard3
dan sebelum pengambilan foto HP. Evaluasi tiga kelas bersama hanya ditambahkan
jika daftar hard3 sudah tersedia.
Gunakan hasil validation untuk keputusan training. Hasil test hanya dipakai untuk
pelaporan dan analisis kegagalan yang diminta tugas. Lakukan inference untuk
enam checkpoint melalui mode `predict`, tanpa mengulang training.


### 13.1 Konfigurasi test evaluation

Simpan evaluasi tiga kelas yang sama dari checkpoint semua kelas maupun hard3.


In [ ]:
register_cell('cell-117', 'final_guard')
def save_common_class_evaluation(split, data, predictions):
    selected = load_hard3()
    assert selected is not None, "Daftar hard3 belum ditetapkan."
    summary, per_class = evaluate_predictions(data, predictions, selected["category_ids"])
    record = {**RUN_CONFIG, **summary, "split": split, "scope": "common_hard3",
              "images": len(data["images"]), "checkpoint": str(CHECKPOINT),
              "evaluation_category_ids": selected["category_ids"]}
    write_json(RUN_DIR / f"metrics_{split}_hard3.json", record)
    per_class.to_csv(RUN_DIR / f"per_class_{split}_hard3.csv", index=False)
    return record

if FINAL_TEST and needs_model:
    assert MODE in {"train", "predict"}, "FINAL_TEST hanya untuk mode train/predict."
    assert not SMOKE_TEST, "Hasil smoke test tidak boleh dipakai untuk evaluasi akhir."
    if SCENARIO == "hard3":
        assert load_hard3() is not None, "Tetapkan tiga kelas dari validasi terlebih dahulu."


### 13.2 Roboflow test evaluation

Ekspor dan evaluasi test asli menggunakan checkpoint yang sudah dipilih dari validation.


In [ ]:
register_cell('cell-119', 'evaluate_test')
if FINAL_TEST and needs_model:
    test_data = full_coco["test"]
    test_predictions = predict_dataset(FULL_COCO / "test", test_data, predict_one)
    rows = [save_evaluation("test", test_data, test_predictions)]
    if load_hard3() is not None:
        rows.append(save_common_class_evaluation("test", test_data, test_predictions))
    write_json(RUN_DIR / "evaluation_test_manifest.json", {
        "dataset_id": DATASET_ID, "image_ids": [i["id"] for i in test_data["images"]]})
    display(pd.DataFrame(rows)[["model", "scenario", "scope", "mask_ap", "mask_ap50"]])


### 13.3 Personal test evaluation

Evaluasi foto HP dengan protokol yang sama dan simpan identitas test set tersebut.


In [ ]:
register_cell('cell-121', 'evaluate_personal')
if FINAL_TEST and needs_model and personal is not None:
    personal_data = personal[2]
    personal_predictions = predict_dataset(personal[1], personal_data, predict_one)
    rows = [save_evaluation("personal", personal_data, personal_predictions),
            save_common_class_evaluation("personal", personal_data, personal_predictions)]
    photo_hashes = {i["file_name"]: hashlib.sha256((personal[1] / i["file_name"]).read_bytes()).hexdigest()
                    for i in personal_data["images"]}
    personal_id = fingerprint({"annotations": personal_data, "images": photo_hashes})
    write_json(RUN_DIR / "evaluation_personal_manifest.json", {
        "dataset_id": personal_id, "images": personal_data["images"]})
    for filename in ["metrics_personal.json", "metrics_personal_hard3.json"]:
        record = read_json(RUN_DIR / filename)
        record["personal_dataset_id"] = personal_id
        write_json(RUN_DIR / filename, record)
    display(pd.DataFrame(rows)[["model", "scenario", "scope", "mask_ap", "mask_ap50"]])


## 14. Sequential training runner


Runner menggunakan satu subprocess per model. Subprocess berikutnya dimulai
setelah proses sebelumnya berakhir, sehingga CUDA context dilepas oleh sistem.
Dependencies tambahan dipasang pada virtual environment per model, dengan
`system_site_packages=True` untuk menggunakan PyTorch bawaan Kaggle.

Status setiap model dicatat sebagai `completed`, `failed`, atau `skipped`.
Run hanya dilewati jika completion marker, configuration signature, dan output
yang diperlukan masih sesuai. Auto-resume menggunakan checkpoint optimizer;
jika training sudah selesai tetapi inference gagal, runner mengulang inference.
KeyboardInterrupt menghentikan sequence dan meminta worker menyimpan recovery.


### 14.1 Worker source collection

Kumpulkan source cell yang diperlukan worker tanpa memasukkan runner itu sendiri.


In [ ]:
register_cell('cell-125', 'sequence_sources')
def collect_worker_sources():
    sources, tags = [], set()
    for record in CELL_REGISTRY.values():
        tag = record["tag"]
        if tag not in {"config", "registry"}:
            sources.append(record["source"])
            tags.add(tag)
        if tag == "evaluate_personal":
            break
    required = {"install", "discover", "yolo_train", "mask_train", "rf_train", "evaluate_test"}
    if not required <= tags:
        raise RuntimeError("Source worker belum lengkap. Jalankan Run All dari awal notebook.")
    return sources

WORKER_SOURCES = collect_worker_sources()
WORKER_SOURCE_ID = fingerprint(WORKER_SOURCES)


### 14.2 Worker configuration signature

Siapkan konfigurasi worker dan signature yang membedakan dataset, model, serta parameter eksperimen.


In [ ]:
register_cell('cell-127', 'sequence_config')
CONFIG_KEYS = [
    "MODE", "MODEL", "MODEL_ORDER", "SKIP_COMPLETED", "AUTO_RESUME", "CONTINUE_ON_ERROR",
    "SCENARIO", "SMOKE_TEST", "FINAL_TEST", "ENABLE_PERSONAL", "SEED", "EPOCHS", "RUN_LABEL",
    "RESUME_CHECKPOINT", "INFERENCE_CHECKPOINT", "RAW_DATASET", "PERSONAL_DATASET",
    "PERSONAL_METADATA", "HARD3_FILE", "PREVIOUS_RESULTS", "YOLO_BATCH", "MASKRCNN_BATCH",
    "RFDETR_BATCH", "YOLO_SIZE", "MASKRCNN_SHORT", "MASKRCNN_LONG", "RFDETR_SIZE",
    "SCORE_FLOOR", "MAX_DETECTIONS", "MIN_VALID_IMAGES", "MIN_TRAIN_IMAGES",
]

def make_job_config(model):
    config = {key: globals()[key] for key in CONFIG_KEYS}
    config.update(MODE="train", MODEL=model, RESUME_CHECKPOINT="", INFERENCE_CHECKPOINT="",
                  EXPECTED_DATASET_ID=DATASET_ID)
    signature = {key: value for key, value in config.items() if key not in {
        "MODE", "RESUME_CHECKPOINT", "INFERENCE_CHECKPOINT", "RAW_DATASET", "PREVIOUS_RESULTS",
        "HARD3_FILE", "SKIP_COMPLETED", "AUTO_RESUME", "CONTINUE_ON_ERROR"}}
    signature.update(dataset_id=DATASET_ID, category_ids=ACTIVE_IDS, source_id=WORKER_SOURCE_ID)
    folder = WORK / "artifacts" / model / SCENARIO / run_name
    return config, fingerprint(signature), folder


### 14.3 Restorasi checkpoint dari Input

Pulihkan output yang diunggah kembali sebagai Input apabila signature sesuai dan direktori run lokal belum tersedia.


In [ ]:
register_cell('cell-129', 'sequence_restore')
def restore_job(signature, folder):
    if (folder / "job_signature.json").exists() or (folder / "run_config.json").exists():
        return
    candidates = []
    for path in find_artifacts("job_signature.json"):
        if path.parent.resolve() == folder.resolve():
            continue
        if read_json(path).get("signature") == signature:
            candidates.append(path.parent)
    if len(candidates) > 1:
        raise ValueError("Terdapat beberapa backup untuk run yang sama. Batasi PREVIOUS_RESULTS ke satu backup.")
    if candidates:
        shutil.copytree(candidates[0], folder, dirs_exist_ok=True)
        print("Output dipulihkan dari Input:", candidates[0])


### 14.4 Completion check dan auto-resume

Periksa completion marker dan pilih resume checkpoint hanya untuk run dengan signature yang sama.


In [ ]:
register_cell('cell-131', 'sequence_resume')
def required_job_files(model):
    names = ["best.pt" if model == "yolo11" else "best.pth",
             "metrics_valid.json", "per_class_valid.csv"]
    if FINAL_TEST:
        names += ["metrics_test.json", "predictions_test.json.gz"]
    return names

def prepare_job(model):
    config, signature, folder = make_job_config(model)
    restore_job(signature, folder)
    previous = folder / "job_signature.json"
    if previous.exists():
        assert read_json(previous)["signature"] == signature, "Konfigurasi berubah; gunakan RUN_LABEL baru."
    elif (folder / "run_config.json").exists():
        raise ValueError(f"Run lama tanpa sequence signature: {folder}. Gunakan RUN_LABEL baru.")
    marker = folder / "completion.json"
    complete = marker.exists() and read_json(marker).get("signature") == signature
    complete = complete and all((folder / p).is_file() and (folder / p).stat().st_size > 0
                                for p in required_job_files(model))
    if SKIP_COMPLETED and complete:
        return config, signature, folder, True
    if AUTO_RESUME:
        sessions = read_json(folder / "training_sessions.json") if (folder / "training_sessions.json").exists() else []
        best = folder / ("best.pt" if model == "yolo11" else "best.pth")
        resume = folder / {"yolo11": "last.pt", "maskrcnn": "last.pth", "rfdetr": "native/last.ckpt"}[model]
        if sessions and sessions[-1]["status"] == "finished" and best.exists():
            config.update(MODE="predict", INFERENCE_CHECKPOINT=str(best))
        elif resume.exists():
            config["RESUME_CHECKPOINT"] = str(resume)
    folder.mkdir(parents=True, exist_ok=True)
    write_json(previous, {"signature": signature, "dataset_id": DATASET_ID})
    return config, signature, folder, False


### 14.5 Worker script generation

Bentuk worker script dari source notebook dan tambahkan completion marker setelah seluruh output model tersedia.


In [ ]:
register_cell('cell-133', 'sequence_worker')
def write_worker(config, signature, folder):
    config_path = folder / "worker_config.json"
    script = folder / "worker.py"
    write_json(config_path, config)
    header = (
        "import json, os, sys\nfrom pathlib import Path\n"
        "os.environ['MPLBACKEND'] = 'Agg'\n"
        "globals().update(json.loads(Path(sys.argv[1]).read_text(encoding='utf-8')))\n"
        "def display(value):\n    print(value.to_string(index=False) if hasattr(value, 'to_string') else value)\n"
    )
    footer = (
        "\nrequired = ['best.pt' if MODEL == 'yolo11' else 'best.pth', 'metrics_valid.json', 'per_class_valid.csv']\n"
        "if FINAL_TEST:\n    required += ['metrics_test.json', 'predictions_test.json.gz']\n"
        "assert all((RUN_DIR / name).is_file() and (RUN_DIR / name).stat().st_size > 0 for name in required)\n"
        f"write_json(RUN_DIR / 'completion.json', {{'signature': {signature!r}, 'status': 'completed'}})\n"
    )
    script.write_text(header + "\n\n".join(WORKER_SOURCES) + footer, encoding="utf-8")
    return script, config_path


### 14.6 Virtual environment dan GPU availability

Sediakan virtual environment per model untuk membatasi perubahan dependencies pada worker terkait.


In [ ]:
register_cell('cell-135', 'sequence_environment')
import tempfile
import venv
import site

def model_python(model):
    folder = Path(tempfile.gettempdir()) / "cpmk3-model-envs" / model
    executable = folder / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if not executable.exists():
        venv.EnvBuilder(with_pip=True, system_site_packages=True).create(folder)
    query = subprocess.check_output([str(executable), "-c", "import sysconfig; print(sysconfig.get_path('purelib'))"], text=True)
    packages = Path(query.strip())
    parents = list(dict.fromkeys(site.getsitepackages() + [
        path for path in sys.path if path.endswith(("site-packages", "dist-packages"))]))
    (packages / "kaggle_parent.pth").write_text("\n".join(parents), encoding="utf-8")
    return executable

def check_sequence_gpu():
    if SMOKE_TEST:
        return
    command = [sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Aktifkan GPU Kaggle sebelum training.'"]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GPU tidak tersedia.")


### 14.7 Worker interruption handler

Tunggu worker berhenti setelah interupsi agar recovery handler memperoleh kesempatan menyimpan checkpoint.


In [ ]:
register_cell('cell-137', 'sequence_interrupt')
import signal

def interrupt_worker(process):
    if process.poll() is not None:
        return
    if os.name == "posix":
        os.killpg(process.pid, signal.SIGINT)
    else:
        process.send_signal(signal.CTRL_BREAK_EVENT)
    try:
        process.wait(timeout=60)
    except subprocess.TimeoutExpired:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()


### 14.8 Subprocess execution dan logging

Stream ringkasan worker ke notebook dan simpan console log lengkap pada direktori model.


In [ ]:
register_cell('cell-139', 'sequence_process')
def execute_worker(python, script, config_path, folder):
    environment = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
    options = {"start_new_session": True} if os.name == "posix" else {
        "creationflags": subprocess.CREATE_NEW_PROCESS_GROUP}
    with (folder / "console_log.txt").open("a", encoding="utf-8") as log:
        process = subprocess.Popen([str(python), "-u", str(script), str(config_path)],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", env=environment, **options)
        last_display = 0.0
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                important = any(word in line for word in ["Traceback", "Error", "checkpoint", "mask_ap"])
                if important or time.monotonic() - last_display >= 5:
                    print(line.rstrip()[-500:], flush=True)
                    last_display = time.monotonic()
            return process.wait()
        except BaseException:
            interrupt_worker(process)
            raise
        finally:
            process.stdout.close()


### 14.9 Run status dan recovery export

Simpan status akhir dan lakukan recovery export dari parent process jika worker gagal atau diinterupsi.


In [ ]:
register_cell('cell-141', 'sequence_status')
def finalize_job(record, started):
    record["seconds"] = time.perf_counter() - started
    folder = WORK / "sequence" / SCENARIO / run_name
    write_json(folder / f"status_{record['model']}.json", record)
    if record["status"] in {"failed", "interrupted"} and record.get("artifact_dir"):
        try:
            export_recovery(record["artifact_dir"])
        except Exception as error:
            print("Parent recovery export gagal:", error)


### 14.10 Eksekusi model dan status recording

Eksekusi satu model, verifikasi output, dan simpan status tanpa menghentikan model berikutnya pada ordinary error.


In [ ]:
register_cell('cell-143', 'sequence_job')
def run_model_job(model):
    started = time.perf_counter()
    record = {"model": model, "scenario": SCENARIO, "status": "failed"}
    try:
        config, signature, folder, skip = prepare_job(model)
        record["artifact_dir"] = str(folder)
        if skip:
            record["status"] = "skipped"
        else:
            python = model_python(model)
            script, config_path = write_worker(config, signature, folder)
            code = execute_worker(python, script, config_path, folder)
            marker = folder / "completion.json"
            if code or not marker.exists() or read_json(marker).get("signature") != signature:
                raise RuntimeError(f"Worker {model} gagal dengan exit code {code}. Periksa {folder / 'console_log.txt'}")
            if not all((folder / name).is_file() for name in required_job_files(model)):
                raise RuntimeError(f"Output {model} belum lengkap.")
            record["status"] = "completed"
    except KeyboardInterrupt:
        record["status"] = "interrupted"
        raise
    except Exception as error:
        record["error"] = str(error)
        print(f"{model}: {error}")
    finally:
        finalize_job(record, started)
    return record


### 14.11 Sequential training execution

Jalankan sequence sesuai MODEL_ORDER dan perbarui ringkasan status setelah setiap worker selesai.


In [ ]:
register_cell('cell-145', 'sequence_run')
def run_sequence():
    check_sequence_gpu()
    statuses = []
    for model in MODEL_ORDER:
        print(f"Memulai worker: {MODEL_NAMES[model]}", flush=True)
        record = run_model_job(model)
        statuses.append(record)
        write_json(WORK / "sequence" / SCENARIO / run_name / "sequence_status.json", statuses)
        display(pd.DataFrame(statuses))
        gc.collect()
        if record["status"] == "failed" and not CONTINUE_ON_ERROR:
            raise RuntimeError(record["error"])
    if all(record["status"] == "failed" for record in statuses):
        raise RuntimeError("Seluruh worker gagal. Periksa sequence_status.json dan console_log.txt.")
    return pd.DataFrame(statuses)

if MODE == "train" and MODEL == "all":
    sequence_results = run_sequence()


## 15. Tabel perbandingan dan domain shift


### 15.1 Tabel perbandingan model

Gabungkan hasil run yang sudah disimpan sebagai Input. Nilai AP tetap dalam skala 0 sampai 1.


In [ ]:
register_cell('cell-148', 'comparison')
comparison = pd.DataFrame(collect_metrics())
if not comparison.empty:
    comparison.to_csv(REPORTS / "comparison.csv", index=False)
    columns = ["model", "scenario", "run_label", "split", "scope", "mask_ap", "mask_ap50", "images"]
    display(comparison[columns].sort_values(["model", "scenario", "split", "scope"]))
else:
    print("Belum ada metrik eksperimen utama untuk digabungkan.")


### 15.2 Domain-shift analysis

Bandingkan test asli dan foto HP pada ruang kelas evaluasi yang sama.


In [ ]:
register_cell('cell-150', 'domain_shift')
if not comparison.empty:
    final_rows = comparison[comparison["split"].isin(["test", "personal"])].copy()
    if not final_rows.empty:
        protocols = final_rows[["score_floor", "max_detections"]].drop_duplicates()
        assert len(protocols) == 1, "Protokol prediksi berbeda antar run."
        if "personal_dataset_id" in final_rows:
            personal_ids = final_rows.loc[final_rows["split"] == "personal", "personal_dataset_id"].dropna().unique()
            assert len(personal_ids) <= 1, "Versi foto HP berbeda antar run."
        keys = ["model", "scenario", "run_label", "scope"]
        shift = final_rows.pivot(index=keys, columns="split", values="mask_ap")
        if {"test", "personal"} <= set(shift.columns):
            shift["drop_ap_points"] = (shift["test"] - shift["personal"]) * 100
            shift.to_csv(REPORTS / "domain_shift.csv")
            display(shift)
            shift[["test", "personal"]].plot.bar(figsize=(12, 5), ylabel="Mask AP")
            plt.tight_layout()
            plt.savefig(REPORTS / "domain_shift.png", dpi=130)
            plt.show()


### 15.3 Perbandingan common classes

Tampilkan perbandingan all_classes dan hard3 pada tiga kelas yang identik.


In [ ]:
register_cell('cell-152', 'hard3_comparison')
if not comparison.empty:
    common = comparison.query("scope == 'common_hard3'")
    if not common.empty:
        table = common.pivot(index=["model", "split", "run_label"], columns="scenario", values="mask_ap")
        if {"all_classes", "hard3"} <= set(table.columns):
            table["hard3_gain_ap_points"] = (table["hard3"] - table["all_classes"]) * 100
        table.to_csv(REPORTS / "hard3_comparison.csv")
        display(table)


## 16. Analisis lima foto kegagalan


### 16.1 Failure-analysis configuration

Pilih folder hasil dan ID foto kegagalan setelah meninjau prediksi. Daftar ini diisi berdasarkan foto sendiri.


In [ ]:
register_cell('cell-155', 'failure_config')
FAILURE_RUN = ""
FAILURE_IMAGE_IDS = []
VISUAL_SCORE = 0.25
DIAGNOSTIC_BRIGHTNESS = 1.4
HYPOTHESIS = ""
if MODE == "diagnose":
    assert personal is not None and len(FAILURE_IMAGE_IDS) == 5
    assert len(set(FAILURE_IMAGE_IDS)) == 5 and HYPOTHESIS.strip()
    assert not SMOKE_TEST


### 16.2 Instance matching dan error count

Hitung objek terlewat dan prediksi berlebih dengan pencocokan mask sekelas pada IoU 0,5.


In [ ]:
register_cell('cell-157', 'image_errors')
def image_errors(data, predictions, allowed_ids):
    by_image_gt, by_image_pred = defaultdict(list), defaultdict(list)
    for ann in data["annotations"]:
        if ann["category_id"] in allowed_ids:
            by_image_gt[ann["image_id"]].append(ann)
    for row in predictions:
        if row["score"] >= VISUAL_SCORE and row["category_id"] in allowed_ids:
            by_image_pred[row["image_id"]].append(row)
    rows = []
    for item in data["images"]:
        truth = by_image_gt[item["id"]]
        proposed = sorted(by_image_pred[item["id"]], key=lambda p: -p["score"])
        matched, tp = set(), 0
        for pred in proposed:
            choices = []
            for index, ann in enumerate(truth):
                if index in matched or ann["category_id"] != pred["category_id"]:
                    continue
                rle = polygon_rle(ann["segmentation"], item["height"], item["width"])
                iou = float(mask_utils.iou([pred["segmentation"]], [rle], [0])[0, 0])
                choices.append((iou, index))
            if choices and max(choices)[0] >= 0.5:
                matched.add(max(choices)[1])
                tp += 1
        rows.append({"image_id": item["id"], "file_name": item["file_name"], "tp": tp,
                     "fn": len(truth) - tp, "fp": len(proposed) - tp})
    return pd.DataFrame(rows).assign(errors=lambda frame: frame["fn"] + frame["fp"])


### 16.3 Failure candidates

Urutkan kandidat foto gagal dan siapkan tabel penjelasan yang harus diisi berdasarkan bukti.


In [ ]:
register_cell('cell-159', 'failure_candidates')
failure_predictions = None
if FAILURE_RUN and personal is not None:
    failure_folder = Path(FAILURE_RUN)
    failure_config = read_json(failure_folder / "run_config.json")
    assert failure_config["dataset_id"] == DATASET_ID
    failure_predictions = read_predictions(failure_folder / "predictions_personal.json.gz")
    errors = image_errors(personal[2], failure_predictions, failure_config["category_ids"])
    errors = errors.sort_values("errors", ascending=False)
    display(errors.head(15))
    errors.to_csv(REPORTS / "failure_candidates.csv", index=False)
    if FAILURE_IMAGE_IDS:
        selected = errors[errors["image_id"].isin(FAILURE_IMAGE_IDS)].copy()
        assert len(selected) == 5 and (selected["errors"] > 0).all()
        selected["observed_failure"] = ""
        selected["model_mechanism"] = ""
        selected["hypothesis"] = ""
        selected["experiment_and_evidence"] = ""
        selected["conclusion"] = ""
        notes_path = REPORTS / "failure_notes.csv"
        if not notes_path.exists():
            selected.to_csv(notes_path, index=False)


### 16.4 Failure visualization

Tampilkan ground truth dan prediksi lima foto pilihan tanpa menyatukan instance menjadi mask semantik.


In [ ]:
register_cell('cell-161', 'failure_visuals')
def show_failure(item, predictions, axes):
    source = Image.open(personal[1] / item["file_name"])
    for axis in axes:
        axis.imshow(source)
        axis.axis("off")
    for ann in personal[2]["annotations"]:
        if ann["image_id"] == item["id"]:
            for polygon in ann["segmentation"]:
                xy = np.asarray(polygon).reshape(-1, 2)
                axes[0].fill(xy[:, 0], xy[:, 1], alpha=0.25)
    for row in predictions:
        if row["image_id"] != item["id"] or row["score"] < VISUAL_SCORE:
            continue
        mask = mask_utils.decode(row["segmentation"])
        axes[1].contour(mask, levels=[0.5], linewidths=1)
        ys, xs = np.where(mask)
        if len(xs):
            axes[1].text(xs.mean(), ys.mean(), CLASS_NAMES[row["category_id"]], fontsize=7)
    axes[0].set_title("Ground truth")
    axes[1].set_title("Prediksi")

if failure_predictions is not None:
    for item in personal[2]["images"]:
        if item["id"] in FAILURE_IMAGE_IDS:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            show_failure(item, failure_predictions, axes)
            plt.tight_layout()
            plt.savefig(REPORTS / f"failure_{item['id']}.png", dpi=130)
            plt.show()


## 17. Eksperimen kecil untuk menguji penjelasan kegagalan


Bagian ini menyediakan eksperimen perubahan kecerahan pada lima foto yang sama.
Gunakan jika hipotesis berkaitan dengan pencahayaan. Isi `HYPOTHESIS` terlebih
dahulu. Ground truth tetap sama karena geometri citra tidak berubah. Perubahan
hasil adalah bukti eksperimen terbatas dan belum membuktikan sebab tunggal.
Untuk hipotesis resolusi atau mekanisme lain, sesuaikan eksperimennya dan catat
perubahannya. Bagian ini tidak dipakai memilih checkpoint atau menyetel model.


### 17.1 Diagnostic experiment

Bandingkan prediksi asli dan foto yang kecerahannya diubah pada checkpoint yang sama.


In [ ]:
register_cell('cell-165', 'diagnostic')
if MODE == "diagnose":
    assert predict_one is not None and HYPOTHESIS.strip()
    subset = copy.deepcopy(personal[2])
    subset["images"] = [i for i in subset["images"] if i["id"] in FAILURE_IMAGE_IDS]
    subset["annotations"] = [a for a in subset["annotations"] if a["image_id"] in FAILURE_IMAGE_IDS]
    assert len(subset["images"]) == 5
    before = predict_dataset(personal[1], subset, predict_one)
    after = predict_dataset(personal[1], subset, predict_one, brightness=DIAGNOSTIC_BRIGHTNESS)
    baseline_metrics, _ = evaluate_predictions(subset, before, ACTIVE_IDS)
    changed_metrics, _ = evaluate_predictions(subset, after, ACTIVE_IDS)
    diagnostic = {"hypothesis": HYPOTHESIS, "image_ids": FAILURE_IMAGE_IDS,
                  "brightness": DIAGNOSTIC_BRIGHTNESS, "checkpoint": str(CHECKPOINT),
                  "baseline": baseline_metrics, "changed": changed_metrics}
    write_json(RUN_DIR / "diagnostic.json", diagnostic)
    write_predictions(RUN_DIR / "diagnostic_before.json.gz", before)
    write_predictions(RUN_DIR / "diagnostic_after.json.gz", after)
    per_image = image_errors(subset, before, ACTIVE_IDS).merge(
        image_errors(subset, after, ACTIVE_IDS), on=["image_id", "file_name"], suffixes=("_before", "_after"))
    per_image.to_csv(RUN_DIR / "diagnostic_per_image.csv", index=False)
    display(per_image)
    display(pd.DataFrame([baseline_metrics, changed_metrics], index=["asli", "kecerahan_diubah"]))


## 18. Logbook dan deklarasi AI


### 18.1 Logbook dan AI disclosure

Tambahkan catatan proses yang benar-benar terjadi. Biarkan kosong bila belum ada entri baru.


In [ ]:
register_cell('cell-168', 'logbook')
LOGBOOK_ENTRY = ""
AI_PROMPT = ""
AI_ERROR = ""
AI_CORRECTION = ""

if LOGBOOK_ENTRY.strip():
    path = REPORTS / "logbook.json"
    records = read_json(path) if path.exists() else []
    records.append({"time_utc": datetime.now(timezone.utc).isoformat(),
                    "model": MODEL, "scenario": SCENARIO, "note": LOGBOOK_ENTRY})
    write_json(path, records)
if AI_PROMPT.strip() or AI_ERROR.strip() or AI_CORRECTION.strip():
    path = REPORTS / "ai_usage.json"
    records = read_json(path) if path.exists() else []
    records.append({"prompt": AI_PROMPT, "error": AI_ERROR, "correction": AI_CORRECTION})
    write_json(path, records)


Notebook ini disusun dengan bantuan AI atas permintaan pengguna. Prompt utama:

> Gabungkan persiapan data, training, dan evaluasi menjadi satu notebook untuk
> Kaggle. Pecah kode menjadi beberapa cell, beri keterangan singkat berbahasa
> Indonesia sebelum setiap cell, gunakan sedikit komentar dan tanpa emoji.

Lengkapi prompt implementasi dan catatan penggunaan AI lain yang benar-benar
dipakai. Ketentuan tugas juga meminta minimal satu kasus nyata kesalahan atau
ketidakcocokan saran AI beserta perbaikannya; jangan mengarang kasus tersebut.


## 19. Menyimpan dan mengunduh hasil


### 19.1 Output archive

Buat arsip checkpoint dan laporan run saat ini untuk diunduh dari Output Kaggle.


In [ ]:
register_cell('cell-172', 'archive')
write_json(WORK / "run_summary.json", {"mode": MODE, "model": MODEL,
           "scenario": SCENARIO, "run_label": run_name, "dataset_id": DATASET_ID,
           "smoke_test": SMOKE_TEST, "personal_enabled": ENABLE_PERSONAL,
           "personal_available": personal is not None,
           "final_test_requested": FINAL_TEST})
archive_base = WORK.parent / f"cpmk3_{MODEL}_{SCENARIO}_{run_name}"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=WORK,
                              base_dir="artifacts") if (WORK / "artifacts").exists() else None
reports_archive = shutil.make_archive(str(WORK.parent / "cpmk3_reports"), "zip",
                                      root_dir=WORK, base_dir="reports")
print("Arsip model:", archive)
print("Arsip laporan:", reports_archive)


Simpan versi notebook beserta outputnya, kemudian unduh arsip dari Output Kaggle.
File `.ipynb` saja tidak memuat checkpoint eksternal. Pertahankan
`run_config.json` di samping `best.pt`, `best.pth`, atau checkpoint resume.
Pada sesi berikutnya, tambahkan output run sebelumnya sebagai Input agar
section perbandingan dapat menemukan hasil semua model.

RF-DETR menyimpan log training asli di direktori `native`, termasuk TensorBoard.
YOLO dan Mask R-CNN menyediakan CSV. Checkpoint resume RF-DETR adalah
`last.ckpt`; checkpoint terbaik digunakan untuk inference. Checkpoint terakhir
YOLO disalin sebelum proses pembersihan optimizer pada akhir training.

Kelengkapan tugas tetap perlu diperiksa: 40 foto sendiri, minimal 10 kondisi
sulit, bukti anotasi dan metadata, enam training utama, analisis domain shift,
lima foto kegagalan, eksperimen kecil, logbook, dan deklarasi AI.

## Referensi

- [Dataset v5](https://universe.roboflow.com/panopticindoor/panoptic-indoor-segmentation/dataset/5), CC BY 4.0.
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/).
- [Format YOLO Segmentation](https://docs.ultralytics.com/datasets/segment/).
- [Mask R-CNN Torchvision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn.html).
- [RF-DETR training](https://rfdetr.roboflow.com/latest/learn/train/).
- [RF-DETR parameter](https://rfdetr.roboflow.com/latest/learn/train/training-parameters/).
- [COCOeval](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).
- [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).
